# HW1 - 2D Flow Matching

Question: how can a model learn to move noise into data?

Sources to inspect first:

- TorchCFM 2D tutorials: https://github.com/atong01/conditional-flow-matching/tree/main/examples/2D_tutorials
- Meta flow_matching examples: https://github.com/facebookresearch/flow_matching/tree/main/examples
- Flow Matching paper: https://arxiv.org/abs/2210.02747

This notebook intentionally uses a tiny MLP instead of a framework abstraction.


In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
def sample_moons(n, noise=0.06):
    # Small local version of two moons so the notebook has no sklearn dependency.
    n1 = n // 2
    n2 = n - n1
    t1 = torch.rand(n1) * math.pi
    t2 = torch.rand(n2) * math.pi
    x1 = torch.stack([torch.cos(t1), torch.sin(t1)], dim=1)
    x2 = torch.stack([1 - torch.cos(t2), 0.5 - torch.sin(t2)], dim=1)
    x = torch.cat([x1, x2], dim=0)
    x = x + noise * torch.randn_like(x)
    x = (x - x.mean(0)) / x.std(0)
    return x

data = sample_moons(4096)
noise = torch.randn_like(data)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.scatter(noise[:,0], noise[:,1], s=3); plt.title("source noise"); plt.axis("equal")
plt.subplot(1, 2, 2); plt.scatter(data[:,0], data[:,1], s=3); plt.title("target data"); plt.axis("equal")
plt.show()


## Core Idea

Pick a noise point `x0` and a data point `x1`.

For the simplest straight path:

```text
x_t = (1 - t) * x0 + t * x1
velocity = x1 - x0
```

The model sees `(x_t, t)` and learns the velocity. Sampling starts from noise and follows predicted velocity.


In [ ]:
class TimeMLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        if t.ndim == 1:
            t = t[:, None]
        return self.net(torch.cat([x, t], dim=1))

model = TimeMLP().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)


In [ ]:
def train_flow(steps=3000, batch_size=512):
    losses = []
    for step in range(steps):
        x1 = sample_moons(batch_size).to(device)
        x0 = torch.randn_like(x1)
        t = torch.rand(batch_size, 1, device=device)
        xt = (1 - t) * x0 + t * x1
        target_v = x1 - x0
        pred_v = model(xt, t.squeeze(1))
        loss = F.mse_loss(pred_v, target_v)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % 500 == 0:
            print(step, loss.item())
    return losses

losses = train_flow()
plt.plot(losses); plt.title("flow matching loss"); plt.show()


In [ ]:
@torch.no_grad()
def sample_flow(n=2048, steps=80, keep_trajectory=False):
    x = torch.randn(n, 2, device=device)
    traj = [x.cpu()]
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((n,), i / steps, device=device)
        v = model(x, t)
        x = x + dt * v
        if keep_trajectory and i % max(1, steps // 8) == 0:
            traj.append(x.cpu())
    return (x.cpu(), traj) if keep_trajectory else x.cpu()

samples, traj = sample_flow(keep_trajectory=True)
plt.figure(figsize=(5, 5))
plt.scatter(samples[:,0], samples[:,1], s=3)
plt.title("samples after following learned velocity")
plt.axis("equal")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(traj), figsize=(3 * len(traj), 3))
for ax, points, i in zip(axes, traj, range(len(traj))):
    ax.scatter(points[:,0], points[:,1], s=2)
    ax.set_title(f"step {i}")
    ax.axis("equal")
    ax.axis("off")
plt.show()


## Comparison Notes

- Flow matching target: velocity.
- Sampler: Euler updates through time.
- Debug signal: trajectories should smoothly move from noise into data.
- Failure mode: too few steps or too small a model gives blurry/incorrect point clouds.
